## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-20-53-49 +0000,nypost,Famed Napa wine guru and ‘Somm’ doc breakout s...,https://nypost.com/2026/11/22/us-news/napa-win...
1,2026-11-22-20-51-02 +0000,nypost,Earthquake strikes California on San Andreas F...,https://nypost.com/2026/11/22/us-news/earthqua...
2,2026-11-22-20-50-18 +0000,nypost,California mall goes to war with famous chef o...,https://nypost.com/2026/11/22/us-news/stanford...
3,2026-11-22-20-47-00 +0000,wsj,Opinion | America Hits the Brakes on AI,https://www.wsj.com/opinion/america-hits-the-b...
4,2026-11-22-20-45-14 +0000,chitri,Okinawa celebrates restored Shuri Castle hall ...,https://www.chicagotribune.com/2026/11/22/okin...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2744/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
87,trump,21
11,california,10
92,new,9
80,house,9
107,deal,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
88,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...,51
119,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...,47
53,2026-11-22-17-58-17 +0000,nypost,Former NYC mayor Bill de Blasio warns Dems not...,https://nypost.com/2026/11/22/us-news/de-blasi...,43
153,2026-11-22-06-43-17 +0000,nypost,Mike Waltz warns Russia faces ‘unsustainable’ ...,https://nypost.com/2026/11/22/us-news/russia-f...,42
68,2026-11-22-17-16-19 +0000,nypost,Chaos strikes California airports as Delta has...,https://nypost.com/2026/11/22/us-news/major-de...,41


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
88,51,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
68,41,2026-11-22-17-16-19 +0000,nypost,Chaos strikes California airports as Delta has...,https://nypost.com/2026/11/22/us-news/major-de...
70,39,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
39,37,2026-11-22-18-42-39 +0000,wapo,House Democrats will start next Congress with ...,https://www.washingtonpost.com/politics/2026/1...
131,32,2026-11-22-10-00-00 +0000,wsj,Richer and more adventurous than their predece...,https://www.wsj.com/personal-finance/retiremen...
97,31,2026-11-22-13-17-40 +0000,cbc,Fire at Indonesian candy factory leaves 3 dead...,https://www.cbc.ca/news/world/indonesian-candy...
119,26,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...
10,25,2026-11-22-20-28-09 +0000,bbc,Explosion on Russian ship at Libya port caused...,https://www.bbc.co.uk/news/articles/cr894gjz0j...
56,25,2026-11-22-17-57-36 +0000,cbc,UN-affiliated athlete swims in dangerous Strai...,https://www.cbc.ca/news/world/un-endurance-swi...
25,22,2026-11-22-19-41-02 +0000,wapo,Democrat Chuck Hubbard defeats GOP incumbent i...,https://www.washingtonpost.com/politics/2026/1...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
